In [3]:
# EXEMPLO — CLASSIFICAÇÃO SUPERVISIONADA COM INPUT MANUAL
# Contexto: Computação Forense - Detecção de login suspeito

# !pip install scikit-learn  # (se necessário)

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(42)

# =============================
# 1) Gerando base sintética para treinar o modelo
# =============================

n = 1200

df = pd.DataFrame({
    "failed_attempts_24h": rng.integers(0, 12, size=n),
    "hour_of_day": rng.integers(0, 24, size=n),
    "new_country": rng.integers(0, 2, size=n),
    "ip_external": rng.integers(0, 2, size=n),
    "vpn_or_proxy": rng.integers(0, 2, size=n),
    "device_new": rng.integers(0, 2, size=n),
    "distance_km": rng.normal(50, 80, size=n).clip(0, 2000)
})

score = (
    0.35 * df["failed_attempts_24h"] +
    0.9  * df["new_country"] +
    0.7  * df["vpn_or_proxy"] +
    0.6  * df["device_new"] +
    0.5  * df["ip_external"] +
    0.001 * df["distance_km"] +
    0.25 * ((df["hour_of_day"] <= 5) | (df["hour_of_day"] >= 23)).astype(int)
)

y = ((score + rng.normal(0, 0.6, size=n)) > 2.2).astype(int)
X = df.copy()

# =============================
# 2) Treinamento do modelo
# =============================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)

# =============================
# 3) Entrada manual de dados
# =============================

print("\n=== INSIRA OS DADOS DO EVENTO DE LOGIN ===")

failed_attempts_24h = int(input("Falhas de login nas últimas 24h: "))
hour_of_day = int(input("Hora do dia (0-23): "))
new_country = int(input("País novo? (0=Não / 1=Sim): "))
ip_external = int(input("IP externo à rede corporativa? (0=Não / 1=Sim): "))
vpn_or_proxy = int(input("VPN/Proxy detectado? (0=Não / 1=Sim): "))
device_new = int(input("Dispositivo novo? (0=Não / 1=Sim): "))
distance_km = float(input("Distância aproximada do último login (km): "))

evento = pd.DataFrame([{
    "failed_attempts_24h": failed_attempts_24h,
    "hour_of_day": hour_of_day,
    "new_country": new_country,
    "ip_external": ip_external,
    "vpn_or_proxy": vpn_or_proxy,
    "device_new": device_new,
    "distance_km": distance_km
}])

# =============================
# 4) Predição
# =============================

probabilidade = model.predict_proba(evento)[0, 1]
classe = model.predict(evento)[0]

print("\n=== RESULTADO ===")
print(f"Probabilidade de ser SUSPEITO: {probabilidade:.3f}")

if classe == 1:
    print("Classificação final: SUSPEITO")
else:
    print("Classificação final: NORMAL")



=== INSIRA OS DADOS DO EVENTO DE LOGIN ===
Falhas de login nas últimas 24h: 0
Hora do dia (0-23): 10
País novo? (0=Não / 1=Sim): 0
IP externo à rede corporativa? (0=Não / 1=Sim): 1
VPN/Proxy detectado? (0=Não / 1=Sim): 0
Dispositivo novo? (0=Não / 1=Sim): 0
Distância aproximada do último login (km): 12

=== RESULTADO ===
Probabilidade de ser SUSPEITO: 0.013
Classificação final: NORMAL
